# Train EfficientNet-B4 for Chest X-Ray Classification

This notebook trains an `EfficientNet-B4` model on the **NIH ChestX-ray14** dataset to classify 14 distinct lung conditions + "No Finding".

**Objective Metrics:**
- AUC-ROC > 0.82
- Sensitivity > 0.75 for Pneumonia, Effusion
- Specificity > 0.85
- Inference time < 2s on CPU

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
import os
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score, f1_score, classification_report

## 1. Dataset & DataLoader Definition

In [ ]:
class ChestXrayDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.label_cols = [
            'Atelectasis','Consolidation','Infiltration','Pneumothorax',
            'Edema','Emphysema','Fibrosis','Effusion','Pneumonia',
            'Pleural Thickening','Cardiomegaly','Nodule','Mass','Hernia','No Finding'
        ]
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['Image Index'])
        img = Image.open(img_path).convert('RGB')
        
        if self.transform:
            img = self.transform(img)
            
        labels = torch.FloatTensor([row[col] for col in self.label_cols])
        return img, labels

## 2. Model Initialization

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 15)
model = model.to(DEVICE)

## 3. Training Loop

In [ ]:
EPOCHS = 20
BATCH_SIZE = 32
LR = 1e-4

criterion = nn.BCEWithLogitsLoss() # For multi-label classification
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for imgs, labels in tqdm(loader, desc="Training"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Evaluating"):
            imgs = imgs.to(DEVICE)
            outputs = torch.sigmoid(model(imgs)).cpu().numpy()
            all_preds.extend(outputs)
            all_labels.extend(labels.numpy())
            
    # Calculate ROC-AUC
    try:
        auc = roc_auc_score(all_labels, all_preds, average="macro")
        print(f"Macro ROC-AUC: {auc:.4f}")
    except ValueError:
        print("ROC-AUC calculation skipped (requires both classes in y_true)")
        
    return all_preds, all_labels

In [ ]:
# TODO: Load DataFrames (train_df, val_df) and populate DataLoaders
# train_loader = DataLoader(ChestXrayDataset(...), batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(ChestXrayDataset(...), batch_size=BATCH_SIZE)

# for epoch in range(EPOCHS):
#     loss = train_epoch(model, train_loader, optimizer, criterion)
#     evaluate(model, val_loader)
#     scheduler.step()

# torch.save(model.state_dict(), '../backend/models/weights/efficientnet_chestxray.pth')